# cockpit-jax: record a diverging run, then replay it

A self-contained example on a toy problem: a small MLP fitting a 2D function under a
two-term loss (a fit term and a first-derivative penalty, so the two terms have
different gradient scales), trained by Adam on a learning rate that ramps up until the
run blows up.

1. train with a `FlightRecorder` attached, which dumps the last few full states when
   the loss spikes
2. load the bundle from before the spike and step it forward by hand, with probes on
3. look at one step in detail: sharpness, the eigenvalue spectrum, gradient noise,
   per-sample maps, the loss along the step taken
4. branch the same state under three interventions and compare on identical batches

Nothing here depends on any library beyond JAX, optax, Orbax and matplotlib.

In [ ]:
import os
from typing import Any

os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_deterministic_ops=true"

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import struct

import cockpit_jax as cj
from cockpit_jax import probes as P, interventions as I, plots as V

jax.config.update("jax_enable_x64", True)
print(jax.devices())

## 1. Problem, model and step

`state` only has to be a pytree with the fields the tools touch: `params` for the
probes, `loss_weights` for `interventions.set_loss_weight`, and `step` as the step
counter. A `flax.struct.dataclass` gives `.replace(...)` for free.

In [ ]:
DOM = np.array([[-1.0, 1.0], [-1.0, 1.0]])
target = lambda z: jnp.sin(3.0 * z[..., 0]) * jnp.cos(3.0 * z[..., 1])

sampler = {"key": jax.random.PRNGKey(0)}          # host-side: must be saved to replay


def next_batch(n=512):
    """batch[:, :2] = inputs, batch[:, 2] = target. One array, so the plotting and
    intervention helpers take it directly."""
    sampler["key"], k = jax.random.split(sampler["key"])
    z = jax.random.uniform(k, (n, 2), minval=jnp.asarray(DOM[:, 0]), maxval=jnp.asarray(DOM[:, 1]))
    return jnp.concatenate([z, target(z)[:, None]], axis=1)


def init_params(key, widths=(2, 64, 64, 1)):
    ps = []
    for k, (i, o) in zip(jax.random.split(key, len(widths) - 1), zip(widths[:-1], widths[1:])):
        ps.append({"W": jax.random.normal(k, (i, o)) * jnp.sqrt(2.0 / i), "b": jnp.zeros((o,))})
    return ps


def u(params, z):
    for layer in params[:-1]:
        z = jnp.tanh(z @ layer["W"] + layer["b"])
    return (z @ params[-1]["W"] + params[-1]["b"])[..., 0]


@struct.dataclass
class TrainState:
    params: Any
    opt_state: Any
    step: int
    loss_weights: dict


def terms_fn(params, state, batch):
    """Two terms with different gradient scales."""
    z, y = batch[:, :2], batch[:, 2]
    pred = u(params, z)
    dudx = jax.vmap(jax.grad(lambda zz: u(params, zz[None])[0]))(z)[:, 0]
    return {"fit": jnp.mean((pred - y) ** 2), "smooth": jnp.mean(dudx ** 2)}


def loss_fn(params, state, batch):
    d = terms_fn(params, state, batch)
    return sum(state.loss_weights[k] * d[k] for k in d)


def sample_loss_fn(params, state, batch):
    """(N,) loss of each sample, for the per-sample gradient probe."""
    z, y = batch[:, :2], batch[:, 2]
    return (u(params, z) - y) ** 2

In [ ]:
# the learning rate ramps up, so this run is meant to blow up partway through
LR = optax.linear_schedule(init_value=3e-3, end_value=0.3, transition_steps=3000)
tx = optax.adam(LR)


def make_step_fn(tx):
    @jax.jit
    def raw(state, batch):
        loss, grads = jax.value_and_grad(loss_fn)(state.params, state, batch)
        updates, opt_state = tx.update(grads, state.opt_state, state.params)
        params = optax.apply_updates(state.params, updates)
        return state.replace(params=params, opt_state=opt_state, step=state.step + 1), loss

    def step_fn(state, host, batch):
        """(state, host, batch) -> (state, host, metrics), the signature Replay wants."""
        state, loss = raw(state, batch)
        return state, host, {"loss": loss}
    return step_fn


step_fn = make_step_fn(tx)

params0 = init_params(jax.random.PRNGKey(1))
state0 = TrainState(params=params0, opt_state=tx.init(params0), step=0,
                    loss_weights={"fit": 1.0, "smooth": 0.1})

## 2. Train with the recorder attached

`host` is everything the step depends on that is not inside the state. Here that is
the sampler key; in a real run it is also any Python-side controller or counter.
Without it the replay draws different batches and is a different trajectory.

In [ ]:
# watch the gradient norm too: it usually spikes a few steps before the loss does
rec = cj.recorder.FlightRecorder("runs/flight", keep=6, every=50, spike=10.0,
                                 min_history=50, watch={"grad_norm": 5.0})
gnorm = jax.jit(lambda st, b: cj.tree.tree_norm(jax.grad(loss_fn)(st.params, st, b)))

state = state0
state, _, _ = step_fn(state, None, next_batch())      # warmup: this step is the compile
state = state0

for step in range(4000):
    batch = next_batch()
    aux = {"grad_norm": float(gnorm(state, batch))}
    state, _, metrics = step_fn(state, None, batch)
    if rec(step, state, float(metrics["loss"]), host={"key": sampler["key"]}, aux=aux):
        print("bundle written at step", step, "--", rec.reason)
        break
    if step % 200 == 0:
        print(step, float(metrics["loss"]))

## 3. Replay from the last clean state

`Replay` holds every state it steps through, so `state_at` and `rewind` work. Branches
share one batch cache, so every branch sees identical data.

In [ ]:
saved = sorted(int(d.split("_")[1]) for d in os.listdir("runs/flight") if d.startswith("step_"))
print("bundles:", saved)

k0 = saved[0]                                   # the earliest buffered state
state_r, host_r = cj.ckpt.load_bundle("runs/flight/step_{}".format(k0), state0)
sampler["key"] = jnp.asarray(host_r["key"])     # so the replay draws the run's batches

probes = {
    "update": P.probe_update(),
    "grad": P.probe_grad(loss_fn),
    "terms": P.probe_terms(terms_fn),
    "curv": P.probe_step_curvature(loss_fn),
    "alpha": P.probe_alpha(loss_fn),
    "dist": P.probe_distance(state_r.params),
    # the heavy two: drop them for a long replay
    "noise": P.probe_noise(sample_loss_fn, chunk=128),
    "sharp": P.probe_sharpness(loss_fn, jax.random.PRNGKey(7), iters=15),
}
rp = cj.replay.Replay(step_fn, state_r, host_r, batch_fn=next_batch,
                      probes=probes, start_step=k0, name="base")

In [ ]:
rp.step(1)                       # one step, with every probe; re-run this cell freely
# rp.step(300, verbose=False)
# rp.run_until(lambda r: r["loss"] > 100 * rp.records[0]["loss"], max_steps=500, verbose=False)

In [ ]:
rp.step(300, verbose=False)
V.plot_records(rp.records, ["loss", "grad/norm", "upd/ratio", "curv/along_step",
                            "curv/top_eig", "dist/from_ref"])
V.plot_records(rp.records, ["alpha"], logy=False)   # 0 = landed at the fitted minimum
V.plot_records(rp.records, ["noise/norm_test", "noise/inner_test", "noise/ortho_test",
                            "noise/trace_sigma", "noise/mean_gsnr"])
V.plot_prefix(rp.records, "gnorm/")            # per term: fit, smooth
V.plot_prefix(rp.records, "gcos/", logy=False) # do the two terms pull against each other

## 4. One step in detail

`state_at(k)` is the state *before* step k, and `batch_at(k)` is the batch it used.

In [ ]:
k = rp.k - 1
st, _ = rp.state_at(k)
b = rp.batch_at(k)

lam, v_top, hist = P.top_hessian_eig(loss_fn, st.params, st, b, jax.random.PRNGKey(0))
tr, tr_se = P.hessian_trace(loss_fn, st.params, st, b, jax.random.PRNGKey(1), n_probes=10)
print("top eigenvalue {:.4e}   trace {:.4e} +- {:.1e}".format(lam, tr, tr_se))

# the matching comparison for Adam: the PRECONDITIONED top eigenvalue against 2/lr
lam_p, _, _ = P.top_preconditioned_eig(loss_fn, st.params, st, b,
                                       P.adam_pinv(st.opt_state), jax.random.PRNGKey(2))
print("preconditioned top eigenvalue {:.4e}   2/lr {:.4e}".format(
    lam_p, 2.0 / float(LR(int(st.step)))))

# the k largest algebraic eigenvalues, and the whole spectrum including its negative part
ev, _ = P.top_k_hessian_eigs(loss_fn, st.params, st, b, jax.random.PRNGKey(3), k=5)
print("top-5:", np.array2string(ev, precision=3))
nodes, wts = P.spectral_density(loss_fn, st.params, st, b, jax.random.PRNGKey(4),
                                n_probes=4, m=30)
V.plot_density(*P.density_curve(nodes, wts), title="eigenvalue density at step {}".format(k))

In [ ]:
# gradient noise at this step: the three tests, CABS, and the two TIC variants
mom = P.grad_moments(sample_loss_fn, st.params, st, b, chunk=128)
print(P.gradient_tests(mom, theta_norm=1.0, theta_inner=1.0, nu_ortho=1.0))
print("tr(Sigma) {:.4e}   mean GSNR {:.4e}".format(mom["trace_sigma"], mom["mean_gsnr"]))
print("CABS batch suggestion {:.1f}".format(
    P.cabs(mom, lr=float(LR(int(st.step))), loss=float(loss_fn(st.params, st, b)))))
hdiag = P.hessian_diagonal(loss_fn, st.params, st, b, jax.random.PRNGKey(5), n_probes=20)
print("TIC diag {:.4e}   TIC trace {:.4e}".format(P.tic_diag(mom, hdiag), P.tic_trace(mom, tr)))
V.plot_values_hist(mom["var"], logx=True, title="per-parameter gradient variance")

pg = P.per_sample_grads(sample_loss_fn, st.params, st, b, chunk=128)
V.plot_points(b, pg["gnorm"], labels=("x1", "x2"), title="per-sample grad norm", log=True)
V.plot_points(b, pg["align"], labels=("x1", "x2"), title="share of the mean gradient")
V.plot_points(b, np.asarray(sample_loss_fn(st.params, st, b)), labels=("x1", "x2"),
              title="per-sample loss", log=True)

V.plot_leaf_bars(P.layer_grad_norms(loss_fn, st.params, st, b), title="grad norm per leaf")
print("non-finite leaves:", P.nonfinite_report(st))
print("first NaN op:", P.locate_nan(loss_fn, st.params, st, b))

In [ ]:
# the loss along the update actually taken: t = 1 is where the step landed
st1, _ = rp.state_at(k + 1)
d = cj.tree.tree_sub(st1.params, st.params)
al = np.linspace(-1.0, 3.0, 41)
V.plot_slice(al, P.loss_slice(loss_fn, st.params, st, b, d, al), title="along step {}".format(k))
print(P.alpha(loss_fn, st, st1, b))        # the cubic fit behind the alpha column

## 5. Branches from one state, on identical batches

Two interventions: a smaller learning rate (a different `step_fn`), and dropping the
5% of samples with the largest loss. Dropping changes N, so the jitted step recompiles
once for that branch.

In [ ]:
rp.rewind(k0)
N = 300

br_lr = rp.branch("lr_x0.1", step_fn=make_step_fn(optax.adam(lambda s: 0.1 * LR(s))))
br_w = rp.branch("smooth_off", state=I.set_loss_weight(rp.state, "smooth", 0.0))

b0 = rp.batch_at(k0)
pl0 = np.asarray(sample_loss_fn(rp.state.params, rp.state, b0))
b_cut, dropped = I.drop_top(b0, pl0, 0.05)
br_cut = rp.branch("drop_worst_5pct")

rp.step(N, verbose=False)
br_lr.step(N, verbose=False)
br_w.step(N, verbose=False)
# the edited batch goes to this branch's first step only: set_batch would change it
# for every branch, since the cache is shared
br_cut.step(1, batch=b_cut, verbose=False)
br_cut.step(N - 1, verbose=False)

V.plot_branches([rp, br_lr, br_w, br_cut], ["loss", "grad/norm", "upd/ratio", "alpha"])

# which leaves actually diverged between two branches
for path, (dist, rel) in cj.tree.diff_leaves(rp.state.params, br_lr.state.params).items():
    print("{:40s} {:.3e}  ({:.2f} relative)".format(path, dist, rel))

In [ ]:
for r in (rp, br_lr, br_w, br_cut):
    r.to_csv("runs/replay_{}_{}.csv".format(k0, r.name))
print("written")